# Van Udyan Plant Image Dataset — iNaturalist Collector

This Google Colab notebook collects **licensed iNaturalist plant photos for the 38 valid scientific names** from the Van Udyan spreadsheet. It saves images into species-named folders and writes attribution/licence metadata to CSV.

**Important**
- This notebook only downloads photos whose photo-level licence is explicitly allowed in the configuration below.
- It excludes photos with missing/unknown licences and excludes NoDerivatives licences by default.
- Check each licence's conditions before redistributing images or publishing a trained model. Attribution metadata is retained.
- The iNaturalist API is for small/medium batches, not bulk scraping. This notebook uses conservative request delays and a cap per species.
- These are iNaturalist observations from a wider geography, not necessarily plants photographed at Van Udyan. Review location and labels before training.


## 1. Configure the collection

Default limit is **30 photos per species** (up to 1,140 images before filtering/deduplication). You can lower this if you want a quick first run. The notebook uses the current iNaturalist API v1.

In [ ]:
import os, re, time, json, csv, hashlib, mimetypes
from pathlib import Path
from urllib.parse import urlparse
import requests
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

# Output files in the Colab runtime. Download the ZIP at the end, or change this
# to a folder under your mounted Google Drive if you want persistent storage.
OUTPUT_DIR = Path("/content/van_udyan_inaturalist")
IMAGE_DIR = OUTPUT_DIR / "images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

MAX_IMAGES_PER_SPECIES = 30   # Increase only if needed; keep collection modest.
OBSERVATIONS_PER_PAGE = 100
REQUEST_DELAY_SECONDS = 1.1   # Respect iNaturalist API recommended practice.
TIMEOUT_SECONDS = 30

# Conservative allow-list for a non-commercial academic project.
# No licence / unknown licence and NoDerivatives licences are excluded.
ALLOWED_PHOTO_LICENSES = {
    "cc0", "cc-by", "cc-by-nc", "cc-by-sa", "cc-by-nc-sa"
}

API_BASE = "https://api.inaturalist.org/v1"
HEADERS = {
    "User-Agent": "VanUdyanPlantClassificationStudentProject/1.0 (academic dataset collection)"
}

SPECIES = [
    "Aegle marmelos",
    "Alstonia scholaris",
    "Azadirachta indica",
    "Bauhinia racemosa",
    "Bauhinia variegata",
    "Bombax ceiba",
    "Butea monosperma",
    "Careya arborea",
    "Citrus aurantiifolia",
    "Cordia dichotoma",
    "Cordia sebestena",
    "Dalbergia latifolia",
    "Dalbergia sissoo",
    "Erythrina variegata",
    "Ficus benghalensis",
    "Ficus religiosa",
    "Gmelina arborea",
    "Grewia tiliifolia",
    "Hibiscus rosa-sinensis",
    "Madhuca longifolia",
    "Melia azedarach",
    "Melia dubia",
    "Mimusops elengi",
    "Mangifera indica",
    "Nerium oleander",
    "Plumeria rubra",
    "Pongamia pinnata",
    "Psidium guajava",
    "Salix tetrasperma",
    "Sterculia urens",
    "Swietenia macrophylla",
    "Syzygium cumini",
    "Tectona grandis",
    "Terminalia arjuna",
    "Terminalia bellirica",
    "Terminalia catappa",
    "Thespesia populnea",
    "Ziziphus mauritiana",
]

print("Species to search:", len(SPECIES))
assert len(SPECIES) == 38, "Check the species list before running."
print("Allowed photo licences:", sorted(ALLOWED_PHOTO_LICENSES))
print("Output folder:", OUTPUT_DIR)


## 2. Helper functions

The script looks up each scientific name in the taxon API, then requests observations with photos for the resolved taxon. It checks the **photo's own licence**, not just the observation's licence. It records observation URL, photo URL, photographer/observer, licence, date, and available location fields.

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)

def api_get(endpoint, params=None, retries=5):
    url = f"{API_BASE}/{endpoint.lstrip('/')}"
    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=TIMEOUT_SECONDS)
            if response.status_code == 429:
                wait = min(30, 2 ** attempt)
                print(f"Rate limited; waiting {wait}s...")
                time.sleep(wait)
                continue
            response.raise_for_status()
            time.sleep(REQUEST_DELAY_SECONDS)
            return response.json()
        except requests.RequestException as exc:
            if attempt == retries - 1:
                print(f"Request failed for {url}: {exc}")
                return None
            time.sleep(min(30, 2 ** attempt))
    return None

def safe_name(text):
    text = text.strip().replace(" ", "_")
    return re.sub(r"[^A-Za-z0-9_.-]", "", text)

def resolve_taxon(scientific_name):
    data = api_get("taxa", {"q": scientific_name, "per_page": 10})
    if not data or not data.get("results"):
        return None
    results = data["results"]
    # Prefer exact scientific-name match; do not silently choose a near match.
    exact = [
        t for t in results
        if (t.get("name") or "").casefold() == scientific_name.casefold()
    ]
    candidates = exact or []
    if not candidates:
        return None
    # Prefer species rank if more than one exact-name result appears.
    candidates.sort(key=lambda t: (t.get("rank") != "species", t.get("is_active") is False))
    return candidates[0]

def licence_code(photo):
    code = photo.get("license_code")
    return str(code).strip().lower() if code else ""

def get_photo_url(photo):
    # Prefer the URL provided by the API; replace square thumbnail with medium
    # only for standard iNaturalist URL patterns.
    url = photo.get("url") or ""
    if not url:
        return ""
    return re.sub(r"/(square|small|thumb)\.", "/medium.", url)

def download_image(url, dest):
    try:
        r = session.get(url, timeout=TIMEOUT_SECONDS, stream=True)
        r.raise_for_status()
        content_type = r.headers.get("Content-Type", "").split(";")[0].lower()
        if not content_type.startswith("image/"):
            r.close()
            return False, "not_image"
        temp_path = dest.with_suffix(dest.suffix + ".part")
        with open(temp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 64):
                if chunk:
                    f.write(chunk)
        r.close()
        if not temp_path.exists() or temp_path.stat().st_size < 1000:
            temp_path.unlink(missing_ok=True)
            return False, "file_too_small"
        temp_path.replace(dest)
        return True, content_type
    except Exception as exc:
        return False, str(exc)[:200]


## 3. Collect licensed photos

Run this cell once. It will create one subfolder per species and stop after the configured number of successfully downloaded images per species. It may take several minutes. Some species may return fewer images, or none, under the selected licence rules.

In [ ]:
metadata_rows = []
summary_rows = []
seen_photo_ids = set()

for scientific_name in tqdm(SPECIES, desc="Species"):
    taxon = resolve_taxon(scientific_name)
    if not taxon:
        summary_rows.append({
            "scientific_name": scientific_name,
            "taxon_id": None,
            "downloaded": 0,
            "status": "Exact taxon match not found"
        })
        continue

    taxon_id = taxon.get("id")
    species_dir = IMAGE_DIR / safe_name(scientific_name)
    species_dir.mkdir(parents=True, exist_ok=True)
    downloaded = 0
    examined_observations = 0
    page = 1
    status = "OK"

    while downloaded < MAX_IMAGES_PER_SPECIES:
        params = {
            "taxon_id": taxon_id,
            "photos": "true",
            "per_page": OBSERVATIONS_PER_PAGE,
            "page": page,
            "order_by": "created_at",
            "order": "desc",
        }
        data = api_get("observations", params)
        if not data or not data.get("results"):
            break

        results = data["results"]
        if not results:
            break

        for obs in results:
            examined_observations += 1
            obs_id = obs.get("id")
            obs_url = f"https://www.inaturalist.org/observations/{obs_id}" if obs_id else ""
            taxon_info = obs.get("taxon") or {}
            observer = obs.get("user") or {}
            photos = obs.get("photos") or []

            for photo in photos:
                if downloaded >= MAX_IMAGES_PER_SPECIES:
                    break
                photo_id = photo.get("id")
                lic = licence_code(photo)
                photo_url = get_photo_url(photo)

                # Require explicit approved licence and a URL. Skip duplicates.
                if lic not in ALLOWED_PHOTO_LICENSES or not photo_url:
                    continue
                if photo_id and photo_id in seen_photo_ids:
                    continue

                ext = os.path.splitext(urlparse(photo_url).path)[1].lower()
                if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
                    ext = ".jpg"
                image_id = photo_id or hashlib.sha1(photo_url.encode("utf-8")).hexdigest()[:12]
                filename = f"{safe_name(scientific_name)}_{image_id}{ext}"
                dest = species_dir / filename

                if dest.exists():
                    success, detail = True, "already_downloaded"
                else:
                    success, detail = download_image(photo_url, dest)

                if not success:
                    continue

                if photo_id:
                    seen_photo_ids.add(photo_id)
                downloaded += 1

                metadata_rows.append({
                    "local_path": str(dest),
                    "scientific_name_requested": scientific_name,
                    "inat_taxon_id": taxon_id,
                    "inat_taxon_name": taxon_info.get("name"),
                    "observation_id": obs_id,
                    "observation_url": obs_url,
                    "photo_id": photo_id,
                    "photo_url_downloaded": photo_url,
                    "photo_attribution": photo.get("attribution"),
                    "observer_name": observer.get("name"),
                    "observer_login": observer.get("login"),
                    "photo_license_code": lic,
                    "photo_license_url": (
                        f"https://creativecommons.org/licenses/"
                        if lic.startswith("cc-") else ""
                    ),
                    "observed_on": obs.get("observed_on"),
                    "place_guess": obs.get("place_guess"),
                    "latitude": (obs.get("geojson") or {}).get("coordinates", [None, None])[1]
                                if obs.get("geojson") and obs["geojson"].get("coordinates") else None,
                    "longitude": (obs.get("geojson") or {}).get("coordinates", [None, None])[0]
                                 if obs.get("geojson") and obs["geojson"].get("coordinates") else None,
                    "download_status": detail,
                })

            if downloaded >= MAX_IMAGES_PER_SPECIES:
                break

        if len(results) < OBSERVATIONS_PER_PAGE:
            break
        page += 1

    summary_rows.append({
        "scientific_name": scientific_name,
        "taxon_id": taxon_id,
        "downloaded": downloaded,
        "observations_examined": examined_observations,
        "status": status if downloaded else "No images found under selected licence rules"
    })

    # Save progress during the run, so metadata isn't lost if a later species fails.
    pd.DataFrame(metadata_rows).to_csv(OUTPUT_DIR / "inat_photo_attribution_metadata.csv", index=False)
    pd.DataFrame(summary_rows).to_csv(OUTPUT_DIR / "collection_summary.csv", index=False)

summary_df = pd.DataFrame(summary_rows)
metadata_df = pd.DataFrame(metadata_rows)
print("Collection finished.")
print("Total downloaded:", len(metadata_df))
display(summary_df)


## 4. Review metadata and class counts

Do not train immediately without checking the images. iNaturalist taxon labels can still be wrong, and a photo may show only part of a plant or a flower/fruit rather than the full tree. Review samples from every folder and keep the attribution CSV with the dataset.

In [ ]:
if not metadata_df.empty:
    display(metadata_df.groupby(
        ["scientific_name_requested", "photo_license_code"], dropna=False
    ).size().unstack(fill_value=0))
    display(metadata_df[[
        "scientific_name_requested", "local_path", "observation_url",
        "observer_name", "observer_login", "photo_license_code", "place_guess"
    ]].head(20))
else:
    print("No images were downloaded. Check API access, taxon matches, or the licence allow-list.")


## 5. Create a ZIP for download

The ZIP includes image folders, attribution metadata, and collection summary. Download it before closing the Colab runtime, or edit `OUTPUT_DIR` near the top to save directly to mounted Google Drive.

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/van_udyan_inaturalist_dataset", "zip", root_dir=OUTPUT_DIR)
print("ZIP created:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))

from google.colab import files
files.download(zip_path)


## 6. Before model training

1. Review images and remove visibly incorrect labels or unusable photos.
2. Preserve `inat_photo_attribution_metadata.csv` whenever you use the images.
3. Check each licence's terms, especially attribution, non-commercial use, and ShareAlike obligations.
4. Avoid a random image-level train/test split if multiple photos come from the same observation or near-identical images; group related images together to reduce leakage.
5. iNaturalist photos are supplementary images. For a Van Udyan-specific evaluation, keep your own verified Van Udyan photos as a separate test set where possible.

### API and licence references
- iNaturalist API: https://api.inaturalist.org/v1/docs/
- API recommended practices: https://www.inaturalist.org/pages/api+recommended+practices
- Photo licence guidance: https://help.inaturalist.org/en/support/solutions/articles/151000169918
- Licensed Observation Images dataset guidance: https://help.inaturalist.org/en/support/solutions/articles/151000223300-how-i-download-inaturalist-photos-to-use-for-research-
